# Description
This was a diagnostic check executed on the raw dataset to assess variable missingness and data shape, these diagnostics were reconstructed in 01-02 for safety but can further be ignored. READ ONLY!

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
path = r"WITHOUTSEARCH_orbis_micro_EU_banks_consolidated_1996_2017.xlsx"

In [ ]:
df = pd.read_excel(path, sheet_name=0)

print("Loaded df shape:", df.shape)
print("First columns:", list(df.columns[:15]))

In [ ]:
df.columns = df.columns.astype(str)
df.columns = df.columns.str.replace("\nm", " ", regex=False)
df.columns = df.columns.str.replace(r"\s+", " ", regex=True).str.strip()
df = df.rename(columns={'Unnamed: 0': 'bank_id'})

In [ ]:
missing_codes = ["n.a.", "n.s.", "n.d.", "na", "NA", "N/A", ""]
df = df.replace(missing_codes, np.nan)

In [ ]:
if "bank_id" not in df.columns:
    raise ValueError(
        "Column 'bank_id' not found after cleaning column names. "
        "Run: print(df.columns[:50]) and look for the identifier column."
    )

In [ ]:
# ORBIS wide exports usually have columns like "Total assets m EUR 2016"
# We detect columns that END with a 4-digit year.
year_pattern = re.compile(r"(19|20)\d{2}$")

year_cols = [c for c in df.columns if year_pattern.search(c)]
id_cols = [c for c in df.columns if c not in year_cols]

print("Found year columns:", len(year_cols))
print("Example year columns:", year_cols[:8])

if len(year_cols) == 0:
    # fallback: try columns containing " 2016" anywhere (less strict)
    year_cols = [c for c in df.columns if re.search(r"(19|20)\d{2}", c)]
    id_cols = [c for c in df.columns if c not in year_cols]
    print("Fallback year columns found:", len(year_cols))
    print("Example fallback year cols:", year_cols[:8])
    if len(year_cols) == 0:
        raise ValueError(
            "No year columns found. Your export may not be wide-by-year. "
            "Check df.columns and confirm how years are stored."
        )

In [ ]:
# =========================
# 5) Melt wide -> long
# =========================
long = df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="var_year",
    value_name="value"
)

# Extract year (first 4-digit year occurrence)
long["year"] = long["var_year"].str.extract(r"((?:19|20)\d{2})").astype(int)

# Remove the trailing year from the variable name (prefer end-of-string removal)
long["variable"] = (
    long["var_year"]
    .str.replace(r"\s*((?:19|20)\d{2})\s*$", "", regex=True)
    .str.strip()
)

long = long.drop(columns=["var_year"])

print("\nLong shape:", long.shape)
print("Year range:", long["year"].min(), "-", long["year"].max())
print("Unique variables:", long["variable"].nunique())
print(long[["variable", "year"]].sample(10))


In [ ]:
# =========================
# 6) Pivot long -> bank-year panel
# =========================
bank_id = "bank_id"

panel = long.pivot_table(
    index=[bank_id, "year"],
    columns="variable",
    values="value",
    aggfunc="first"
).reset_index()

# Clean column index name (after pivot)
panel.columns.name = None

print("\nPanel shape:", panel.shape)
print("Panel columns (first 20):", list(panel.columns[:20]))
display(panel.head())


In [ ]:
# =========================
# 7) Create derived variables (example: equity ratio)
# =========================
# Identify the exact names present
cols_lower = {c.lower(): c for c in panel.columns}

def find_col_contains(substring: str):
    substring = substring.lower()
    matches = [c for c in panel.columns if substring in c.lower()]
    return matches

print("\nPossible equity cols:", find_col_contains("total equity"))
print("Possible assets cols:", find_col_contains("total assets"))

In [ ]:
# Try to compute equity_ratio if both exist with 'EUR'
equity_candidates = [c for c in panel.columns if c.lower() == "total equity eur"]
assets_candidates = [c for c in panel.columns if c.lower() == "total assets eur"]

if len(equity_candidates) == 1 and len(assets_candidates) == 1:
    equity_col = equity_candidates[0]
    assets_col = assets_candidates[0]
    panel["equity_ratio"] = panel[equity_col] / panel[assets_col]
    print("\nCreated equity_ratio.")
    print(panel["equity_ratio"].describe())
else:
    print("\nDid not create equity_ratio automatically.")
    print("If your columns have slightly different names, pick from the lists above and compute manually, e.g.:")
    print("panel['equity_ratio'] = panel['<equity col>'] / panel['<assets col>']")


This is very reasonable for banks
Mean / median ≈ 8–9% capital ratio → textbook banking numbers
Upper tail < 1 → fine
Distribution looks exactly like EU banks over this period
So the ratio itself is correct\.

This looks scary but it’s actually not uncommon in raw bank data:
Negative equity can occur due to: accumulated losses, accounting write\-downs, restructuring / resolution
It’s rare \(you have only a few cases\)

In [ ]:
panel.loc[panel["equity_ratio"] < 0, "equity_ratio"] = np.nan

Why this is defensible: 
Negative equity reflects accounting distress or resolution states, not a degree of capitalisation\.
In EWS settings, negative equity: does not have a meaningful continuous interpretation, often reflects post\-distress accounting effects
Treating it as missing avoids: extreme leverage dominating model training, misleading marginal effects in linear benchmarks
This is very common in banking datasets and explicitly recommended in applied work\.

In [ ]:
panel.loc[panel["equity_ratio"] > 1.5, "equity_ratio"] = np.nan


Why this is also standard:
Equity ratios \>100% usually arise from: very small asset denominators, consolidation artefacts, reporting quirks
They are outliers, not economically meaningful capital buffers
Capping \(or removing\) extreme ratios: stabilises estimation, prevents spurious interaction effects, does not materially affect results when incidence is low \(as in your data\)
Many papers either: winsorise, or set implausible ratios to missing\. You chose the most conservative option\.

In [ ]:
panel["equity_ratio"].describe()


Only lost around ~16 observations

Bank capitalisation is proxied using the equity\-to\-assets ratio, constructed from total equity and total assets reported in ORBIS\. Observations with negative equity ratios are set to missing, as negative capital does not admit a meaningful continuous interpretation in early\-warning models and typically reflects post\-distress accounting effects\. In addition, implausibly large ratios exceeding 150% are treated as missing to limit the influence of extreme outliers arising from reporting artefacts\. These adjustments affect only a small number of observations and do not materially alter the effective sample size\.

In [ ]:
(panel[["Total assets EUR", "Total equity EUR", "equity_ratio"]]
 .isna()
 .mean()
 * 100
).round(1)


<5% missingness

Exploratory diagnostics of the ORBIS bank\-level data indicate that missingness for core balance\-sheet variables, including total assets and equity, is limited \(below 5%\)\. The resulting panel remains sufficiently large and informative for the planned lagged early\-warning analysis\. Given this coverage, no imputation is applied\.

In [ ]:
# =========================
# 8) Diagnostics to answer supervisor feedback
# =========================
# (A) Unbalancedness: years per bank
years_per_bank = panel.groupby(bank_id)["year"].nunique()
print("\nYears per bank (summary):")
print(years_per_bank.describe())


## Limiting panel to 2004\-2017

In [ ]:
panel_04017 = panel[(panel["year"] >= 2004) & (panel["year"] <= 2017)].copy()

panel_04017.shape

In [ ]:
# =========================
# DIAGNOSTIC CHECKS (READ-ONLY)
# =========================

print("\n=== DIAGNOSTICS: WIDE EXPORT ===")

asset_cols = [c for c in df.columns if "Total assets EUR" in c.lower()]
print("Asset columns in wide export:", asset_cols)

if asset_cols:
    print(
        "Avg missingness in wide Total assets:",
        df[asset_cols].isna().mean().mean()
    )

print("\n=== DIAGNOSTICS: LONG FORMAT ===")

dup_counts = (
    long.groupby(["bank_id", "year", "variable"])
        .size()
)

print("Max duplicates per (bank_id, year, variable):", dup_counts.max())

if dup_counts.max() > 1:
    print("WARNING: duplicates exist before pivot!")
    print(dup_counts[dup_counts > 1].head())

print("\n=== DIAGNOSTICS: PANEL FORMAT ===")

if "Total assets" in panel.columns:
    print(
        "Missingness Total assets EUR (full panel):",
        panel["Total assets EUR"].isna().mean()
    )
    print(
        "Missingness Total assets EUR (2004–2017):",
        panel_04017["Total assets EUR"].isna().mean()
    )

print("\nMissingness by year (Total assets EUR):")
print(
    panel.groupby("year")["Total assets EUR"]
         .apply(lambda x: x.isna().mean())
)

print("\n=== END DIAGNOSTICS ===")


In [ ]:
years_per_bank_04017 = panel_04017.groupby("bank_id")["year"].nunique()

print("Years per bank (2004–2017):")
print(years_per_bank_04017.describe())


In [ ]:
comparison = pd.DataFrame({
    "Full sample": years_per_bank.describe(),
    "2004–2017": years_per_bank_04017.describe()
})

comparison

Lose 5 banks out of 573, average cover is identical, distribution is unchanged

Restricting the bank\-level sample to the period 2004–2017, which aligns with the availability of key macroeconomic indicators, does not materially affect the structure of the panel\. The number of observed banks decreases only marginally, and average time coverage per bank remains virtually unchanged, with a median of eight observed years\. This indicates that the integration of macro\-level data does not come at the cost of reduced bank\-level coverage\.

In [ ]:
panel["bank_id"].nunique(), panel_04017["bank_id"].nunique()


In [ ]:
# (B) Banks per year
banks_per_year = panel.groupby("year")[bank_id].nunique()
print("\nBanks per year (head/tail):")
print(banks_per_year.head())
print(banks_per_year.tail())

In [ ]:
banks_per_year.describe()

In [ ]:
banks_per_year_04017 = panel_04017.groupby("year")["bank_id"].nunique()

banks_per_year_04017.describe()

In [ ]:
comparison2 = pd.DataFrame({
    "Full sample": banks_per_year.describe(),
    "2004–2017": banks_per_year_04017.describe()
})

comparison2

Exploratory diagnostics of the ORBIS bank\-level data indicate that coverage prior to the early 2000s is limited, with relatively few banks observed in the initial years of the sample\. Restricting the analysis to the period 2004–2017, during which ORBIS coverage becomes substantially denser, does not materially affect average time coverage per bank but significantly improves cross\-sectional depth\. As a result, the empirical analysis focuses on the post\-2004 period, which provides a cleaner and more informative micro\-level panel\.

In [ ]:
banks_by_year = panel_04017.groupby("year")["bank_id"].apply(set)

overlap = {
    year: len(banks_by_year[year] & banks_by_year[year-1]) / len(banks_by_year[year])
    for year in banks_by_year.index if year-1 in banks_by_year
}

pd.Series(overlap).describe()

In [ ]:
# (C) Missingness for a few key vars (auto-pick TA and Equity if present)
key_vars = []
for target in ["total assets m eur", "total equity m eur", "equity_ratio"]:
    for c in panel.columns:
        if c.lower() == target:
            key_vars.append(c)

if key_vars:
    miss = (panel[key_vars].isna().mean() * 100).round(1)
    print("\nMissingness (%):")
    print(miss)
else:
    print("\nNo key vars auto-detected for missingness table. Use panel.columns to select variables manually.")


In [ ]:
(panel[["Total assets EUR", "Total equity EUR", "equity_ratio"]]
 .isna()
 .mean()
 * 100
).round(1)

In [ ]:
miss_04017 = (panel_04017[["Total assets EUR", "Total equity EUR", "equity_ratio"]]
 .isna()
 .mean()
 * 100
).round(1)
print(miss_04017)

In [ ]:
# (D) Optional M&A signal: large TA jumps if Total assets exists
if "Total assets EUR" in panel.columns:
    panel = panel.sort_values([bank_id, "year"])
    panel["ta_growth"] = panel.groupby(bank_id)["Total assets EUR"].pct_change()
    print("\nTop potential M&A/restructuring jumps (ta_growth > 2):")
    display(panel.loc[panel["ta_growth"] > 2, [bank_id, "year", "ta_growth"]].head(20))
else:
    print("\nTotal assets m EUR not found exactly; check panel.columns for the correct TA column name.")


Large year\-on\-year changes in total assets are observed for a small number of banks, consistent with mergers, acquisitions, or major restructuring events\. These cases are relatively infrequent and dispersed across institutions and years, suggesting that such events do not dominate the structure of the bank\-level panel\. Given the focus on systemic banking crises rather than individual bank trajectories, banks are treated as they appear in the database, and no attempt is made to reconstruct pre\-merger entities\. The use of lagged predictors further mitigates the influence of contemporaneous balance\-sheet jumps on crisis prediction\.

In [ ]:
panel_analysis = panel[(panel["year"] >= 2004) & (panel["year"] <= 2017)].copy()

panel_analysis = panel_analysis.sort_values([bank_id, "year"])
panel_analysis["ta_growth"] = (
    panel_analysis.groupby(bank_id)["Total assets EUR"]
    .pct_change(fill_method=None)
)

display(
    panel_analysis.loc[
        panel_analysis["ta_growth"] > 2,
        [bank_id, "year", "ta_growth"]
    ].head(20)
)


In [ ]:
# Select only actual variables (exclude identifiers)
vars_only = panel.columns.difference(["bank_id", "year", "ta_growth"])

missing_all = (
    panel[vars_only]
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

missing_all


missing_04017 = (
    panel_04017[vars_only]
    .isna()
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

missing_04017

missing_compare = pd.DataFrame({
    "Full sample (%)": missing_all,
    "2004–2017 (%)": missing_04017
})

missing_compare


In [ ]:
panel.groupby("year").size()

In [ ]:
panel.shape